# Countdown reinforcement fine-tuning

This notebook compares two reinforcement fine-tuning approaches for the same Countdown task:

1. a `gpt-5.4-mini` model/score grader;
2. a deterministic Python grader.

Both jobs use the checked-in 100-row training and 50-row validation datasets, `Alibaba/qwen3.6-35b-a3b` version `1`, and `GlobalStandard` training. The separate 100-row evaluation file is validated for dataset provenance but is not uploaded for training.

The notebook validates the datasets and graders, configures the model-grader deployment, submits or resumes both jobs, monitors them to terminal state, and summarizes the service validation-reward curves.

> Training and grader deployment capacity can incur charges. Actual results may vary by model version, data, configuration, region availability, and service conditions.

## 1. Configure the project and grader deployment

Copy this cookbook's `.env.template` to a module-local `.env`. Configure a Foundry project that supports the training model and a `gpt-5.4-mini` `DataZoneStandard` deployment in the same Azure AI Services account.

The Azure CLI performs the ARM quota and deployment operations using the identity selected by `az login`. `DefaultAzureCredential` authenticates the Foundry SDK operations and may select the same Azure CLI identity or another configured credential source. Both identities need access to the configured subscription, resource group, account, and project.

By default, `FOUNDRY_GRADER_CAPACITY=max` assigns the grader deployment all currently available matching quota. You may instead set an explicit capacity-unit limit without editing the notebook. The notebook requires the resulting deployment rate limit to be at least `FOUNDRY_GRADER_MIN_TPM` (default `100000` TPM) so the score-model sampling evaluation has sufficient throughput. Review the printed read-only plan, then set `FOUNDRY_ALLOW_GRADER_DEPLOYMENT_UPDATE=true` to authorize creation or update.

If `outputs/submission-state.json` already records the model-grader job, a rerun verifies the grader deployment but does not change its capacity. `outputs/grader-deployment-state.json` records the prior and requested capacity for later reconciliation. The notebook does not automatically restore or delete the deployment because an active job may still require it.

Executing the deployment, upload, and submission cells performs billable operations and submits or resumes two training jobs. Reruns reuse the recorded job IDs when the recipe fingerprint matches.

In [ ]:
import ast
import csv
import hashlib
import io
import json
import math
import operator
import os
import shutil
import subprocess
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential


def find_demo_root():
    current = Path.cwd().resolve()
    candidates = [current, current.parent] if current.name == "notebooks" else [current]
    for candidate in candidates:
        if (
            (candidate / "data" / "preserved" / "countdown-training-rft-100.jsonl").exists()
            and (candidate / "notebooks" / "demo.ipynb").exists()
        ):
            return candidate
    raise FileNotFoundError("Run this notebook from the Countdown cookbook or its notebooks directory.")


def env_bool(name, default=False):
    value = os.getenv(name, "").strip().casefold() or str(default).casefold()
    if value not in {"true", "false"}:
        raise ValueError(f"{name} must be true or false.")
    return value == "true"


def positive_integer(name, default):
    value = os.getenv(name, "").strip() or str(default)
    if not value.isdecimal() or int(value) < 1:
        raise ValueError(f"{name} must be a positive integer.")
    return int(value)


DEMO_ROOT = find_demo_root()
OUTPUT_ROOT = DEMO_ROOT / "outputs"
STATE_PATH = OUTPUT_ROOT / "submission-state.json"
GRADER_DEPLOYMENT_STATE_PATH = OUTPUT_ROOT / "grader-deployment-state.json"


from dotenv import load_dotenv

load_dotenv(DEMO_ROOT / ".env", override=False)


PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "").strip()
SUBSCRIPTION_ID = os.getenv("FOUNDRY_SUBSCRIPTION_ID", "").strip()
RESOURCE_GROUP = os.getenv("FOUNDRY_RESOURCE_GROUP", "").strip()
RESOURCE_LOCATION = os.getenv("FOUNDRY_RESOURCE_LOCATION", "").strip()

MODEL = "qwen3.6-35b-a3b"
MODEL_PUBLISHER = "Alibaba"
MODEL_VERSION = "1"
TRAINING_TYPE = "globalStandard"
GRADER_MODEL = "gpt-5.4-mini"
GRADER_MODEL_VERSION = "2026-03-17"
GRADER_DEPLOYMENT = os.getenv("FOUNDRY_GRADER_DEPLOYMENT", "").strip() or GRADER_MODEL
GRADER_DEPLOYMENT_SKU = "DataZoneStandard"
GRADER_QUOTA_NAME = "OpenAI.DataZoneStandard.gpt-5.4-mini"
GRADER_CAPACITY_SETTING = os.getenv("FOUNDRY_GRADER_CAPACITY", "").strip().casefold() or "max"
GRADER_MIN_TPM = positive_integer("FOUNDRY_GRADER_MIN_TPM", 100_000)
ALLOW_GRADER_DEPLOYMENT_UPDATE = env_bool(
    "FOUNDRY_ALLOW_GRADER_DEPLOYMENT_UPDATE",
    False,
)


print("Project:", PROJECT_ENDPOINT)
print("Region:", RESOURCE_LOCATION)
print("Model:", f"{MODEL_PUBLISHER}/{MODEL}, version {MODEL_VERSION}")
print("Training type:", TRAINING_TYPE)
print("Grader deployment:", GRADER_DEPLOYMENT, GRADER_MODEL, GRADER_DEPLOYMENT_SKU)
print("Grader capacity:", GRADER_CAPACITY_SETTING, f"(minimum {GRADER_MIN_TPM:,} TPM)")

## 2. Validate the exact datasets

In [ ]:
DATASETS = {
    "training": {
        "path": DEMO_ROOT / "data" / "preserved" / "countdown-training-rft-100.jsonl",
        "bytes": 126202,
        "rows": 100,
        "sha256": "5dfdd0dfcbab25119d79ea7e423fde29b22db41db26cc939407c7be41d801307",
    },
    "validation": {
        "path": DEMO_ROOT / "data" / "preserved" / "countdown-validation-rft-50.jsonl",
        "bytes": 63104,
        "rows": 50,
        "sha256": "d6dcfef82e8bf23a7cbd111d1dd9793763f5dd640c3110daba5fa1453638eb9c",
    },
    "evaluation_provenance": {
        "path": DEMO_ROOT / "data" / "preserved" / "countdown-evaluation-100.jsonl",
        "bytes": 5435,
        "rows": 100,
        "sha256": "d433d597e8ebc8f255347be5724591c4b6776bda97fdc5e30ac6ff1d5e4f48dd",
    },
}


def validate_jsonl(label, contract):
    raw_content = contract["path"].read_bytes()
    content = raw_content.replace(b"\r\n", b"\n")
    rows = [json.loads(line) for line in content.splitlines() if line.strip()]
    actual = {
        "raw_bytes": len(raw_content),
        "raw_sha256": hashlib.sha256(raw_content).hexdigest(),
        "normalized_bytes": len(content),
        "rows": len(rows),
        "normalized_sha256": hashlib.sha256(content).hexdigest(),
    }
    normalized = {
        "bytes": actual["normalized_bytes"],
        "rows": actual["rows"],
        "sha256": actual["normalized_sha256"],
    }
    expected = {key: contract[key] for key in ("bytes", "rows", "sha256")}
    if normalized != expected:
        raise ValueError(f"{label} integrity mismatch: expected {expected}, got {actual}")
    for index, row in enumerate(rows, start=1):
        if label == "evaluation_provenance":
            if set(row) != {"item"} or set(row["item"]) != {"target", "nums"}:
                raise ValueError(
                    f"{label} row {index} has unexpected keys: {sorted(row)}"
                )
            item = row["item"]
        else:
            if set(row) != {"messages", "target", "nums"}:
                raise ValueError(
                    f"{label} row {index} has unexpected keys: {sorted(row)}"
                )
            if len(row["messages"]) != 1 or row["messages"][0].get("role") != "user":
                raise ValueError(f"{label} row {index} must contain one user message")
            item = row
        numbers = json.loads(item["nums"])
        if not isinstance(numbers, list) or not numbers:
            raise ValueError(f"{label} row {index} has invalid nums")
        float(item["target"])
    return rows, actual


validated_datasets = {}
for label, contract in DATASETS.items():
    rows, evidence = validate_jsonl(label, contract)
    validated_datasets[label] = rows
    print(label, evidence)

## 3. Validate the shared response and grader contracts

In [ ]:
RESPONSE_FORMAT = json.loads(
    (DEMO_ROOT / "graders" / "preserved" / "countdown-response-schema.json").read_text(
        encoding="utf-8"
    )
)

MODEL_GRADER = {
    "name": "countdown_model_grader",
    "type": "score_model",
    "model": GRADER_MODEL,
    "input": [
        {
            "role": "developer",
            "content": (
                "Grade a Countdown answer. Verify that the JSON output has expression and result, "
                "that the expression uses each input number exactly once, uses only +, -, *, / and "
                "parentheses, that the reported result equals the expression, and that it reaches "
                "the target. Score 5 for an exact valid solution, 4 when off by 1, 3 when off by "
                "2-5, 2 when farther away but otherwise valid, 1 for a result mismatch, and 0 for "
                "invalid syntax or number usage. Return only a score and brief reasoning."
            ),
        },
        {
            "role": "user",
            "content": (
                "target={{item.target}} numbers={{item.nums}} "
                "output={{sample.output_text}}"
            ),
        },
    ],
    "pass_threshold": 5,
    "range": [0.0, 5.0],
}

PYTHON_GRADER_SOURCE = r'''
import ast
import json

OPS = {
    ast.Add: lambda a, b: a + b,
    ast.Sub: lambda a, b: a - b,
    ast.Mult: lambda a, b: a * b,
    ast.Div: lambda a, b: a / b,
}


def evaluate(node, numbers):
    if isinstance(node, ast.Expression):
        return evaluate(node.body, numbers)
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        numbers.append(float(node.value))
        return float(node.value)
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](
            evaluate(node.left, numbers),
            evaluate(node.right, numbers),
        )
    raise ValueError("Only numeric literals and +, -, *, / are allowed")


def grade(sample, item) -> float:
    try:
        expression = sample["output_json"]["expression"]
        used_numbers = []
        calculated = evaluate(ast.parse(expression, mode="eval"), used_numbers)
        expected_numbers = sorted(float(value) for value in json.loads(item["nums"]))
        if sorted(used_numbers) != expected_numbers:
            return 0
        reported = float(sample["output_json"]["result"])
        target = float(item["target"])
        if abs(calculated - reported) > 1e-9:
            return 1
        difference = abs(calculated - target)
        if difference < 1e-9:
            return 5
        if difference <= 1:
            return 4
        if difference <= 5:
            return 3
        return 2
    except (KeyError, TypeError, ValueError, SyntaxError, ZeroDivisionError, json.JSONDecodeError):
        return 0
'''

PYTHON_GRADER = {
    "type": "python",
    "source": PYTHON_GRADER_SOURCE,
}

if MODEL_GRADER["type"] != "score_model" or MODEL_GRADER["range"] != [0.0, 5.0]:
    raise ValueError("Model grader contract changed")
if MODEL_GRADER["pass_threshold"] != 5 or MODEL_GRADER["model"] != "gpt-5.4-mini":
    raise ValueError("Model grader threshold or model changed")
if RESPONSE_FORMAT.get("type") != "json_schema":
    raise ValueError("Expected a JSON-schema response format")

namespace = {}
exec(PYTHON_GRADER_SOURCE, namespace)
grade = namespace["grade"]
grader_tests = [
    (
        "exact",
        {"output_json": {"expression": "(1 + 3) * (2 + 4)", "result": "24"}},
        {"target": "24", "nums": "[1, 2, 3, 4]"},
        5,
    ),
    (
        "result mismatch",
        {"output_json": {"expression": "(1 + 3) * (2 + 4)", "result": "25"}},
        {"target": "24", "nums": "[1, 2, 3, 4]"},
        1,
    ),
    (
        "reused number",
        {"output_json": {"expression": "4 * 4 + 3 + 2", "result": "21"}},
        {"target": "21", "nums": "[1, 2, 3, 4]"},
        0,
    ),
    (
        "unsupported syntax",
        {"output_json": {"expression": "sum([1, 2, 3, 4])", "result": "10"}},
        {"target": "10", "nums": "[1, 2, 3, 4]"},
        0,
    ),
]
for name, sample, item, expected in grader_tests:
    actual = grade(sample, item)
    if actual != expected:
        raise AssertionError(f"{name}: expected {expected}, got {actual}")
    print(name, actual)

## 4. Check or update the grader deployment, then upload the exact 100/50 training inputs

In [ ]:
def run_az_json(*arguments):
    executable = shutil.which("az")
    if not executable:
        raise RuntimeError("Azure CLI is required. Install it and run az login.")
    completed = subprocess.run(
        [executable, *arguments, "--only-show-errors", "-o", "json"],
        capture_output=True,
        text=True,
        timeout=180,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or completed.stdout.strip())
    return json.loads(completed.stdout)


def normalized_location(value):
    return "".join(character for character in value.casefold() if character.isalnum())


def validate_existing_grader(deployment):
    if not deployment:
        return
    model = deployment.get("properties", {}).get("model", {})
    expected = {
        "format": "OpenAI",
        "name": GRADER_MODEL,
        "version": GRADER_MODEL_VERSION,
    }
    actual_sku = deployment.get("sku", {}).get("name")
    if any(model.get(key) != value for key, value in expected.items()):
        raise RuntimeError(
            f"Deployment name collision: {GRADER_DEPLOYMENT} serves "
            f"{model.get('name')} {model.get('version')} ({model.get('format')}). "
            "Choose another FOUNDRY_GRADER_DEPLOYMENT; no resource was changed."
        )
    if actual_sku != GRADER_DEPLOYMENT_SKU:
        raise RuntimeError(
            f"Deployment name collision: {GRADER_DEPLOYMENT} uses SKU {actual_sku}. "
            f"Expected {GRADER_DEPLOYMENT_SKU}; no resource was changed."
        )


def choose_grader_capacity(quota_limit, quota_used, existing_capacity, setting):
    maximum = int(quota_limit) - int(quota_used) + int(existing_capacity)
    minimum_capacity = math.ceil(GRADER_MIN_TPM / 1_000)
    if maximum < minimum_capacity:
        raise RuntimeError(
            f"Available grader capacity is {maximum}, below the {minimum_capacity} "
            f"capacity units required for {GRADER_MIN_TPM:,} TPM."
        )
    if setting == "max":
        return maximum
    if not setting.isdecimal() or int(setting) < minimum_capacity:
        raise ValueError(
            "FOUNDRY_GRADER_CAPACITY must be 'max' or an integer of at least "
            f"{minimum_capacity} capacity units."
        )
    requested = int(setting)
    if requested > maximum:
        raise RuntimeError(
            f"Requested grader capacity {requested} exceeds the available maximum {maximum}."
        )
    return requested


def deployment_token_limit(deployment):
    for rate_limit in deployment.get("properties", {}).get("rateLimits", []):
        if rate_limit.get("key") == "token":
            return int(rate_limit["count"])
    raise RuntimeError("The grader deployment response did not include a token rate limit.")


def grader_job_is_recorded():
    if not STATE_PATH.exists():
        return False
    state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
    return bool(state.get("jobs", {}).get("model-grader"))


def grader_account_name():
    hostname = urlparse(PROJECT_ENDPOINT).hostname or ""
    suffix = ".services.ai.azure.com"
    if not hostname.endswith(suffix):
        raise ValueError(
            "FOUNDRY_PROJECT_ENDPOINT must use the account.services.ai.azure.com format."
        )
    return hostname.removesuffix(suffix)


def get_grader_deployment(account_name):
    deployments = run_az_json(
        "cognitiveservices", "account", "deployment", "list",
        "--name", account_name,
        "--resource-group", RESOURCE_GROUP,
        "--subscription", SUBSCRIPTION_ID,
    )
    deployment = next(
        (item for item in deployments if item.get("name") == GRADER_DEPLOYMENT),
        None,
    )
    if deployment:
        deployment = run_az_json(
            "cognitiveservices", "account", "deployment", "show",
            "--name", account_name,
            "--resource-group", RESOURCE_GROUP,
            "--subscription", SUBSCRIPTION_ID,
            "--deployment-name", GRADER_DEPLOYMENT,
        )
    validate_existing_grader(deployment)
    return deployment


def plan_grader_deployment():
    if not all((PROJECT_ENDPOINT, SUBSCRIPTION_ID, RESOURCE_GROUP, RESOURCE_LOCATION)):
        raise RuntimeError(
            "Set FOUNDRY_PROJECT_ENDPOINT, FOUNDRY_SUBSCRIPTION_ID, "
            "FOUNDRY_RESOURCE_GROUP, and FOUNDRY_RESOURCE_LOCATION."
        )
    account_name = grader_account_name()
    account = run_az_json(
        "cognitiveservices", "account", "show",
        "--name", account_name,
        "--resource-group", RESOURCE_GROUP,
        "--subscription", SUBSCRIPTION_ID,
    )
    if account.get("name") != account_name:
        raise RuntimeError("The configured project endpoint does not match the ARM account.")
    if normalized_location(account.get("location", "")) != normalized_location(RESOURCE_LOCATION):
        raise RuntimeError(
            f"FOUNDRY_RESOURCE_LOCATION={RESOURCE_LOCATION} does not match "
            f"the account location {account.get('location')}."
        )

    usage = run_az_json(
        "cognitiveservices", "usage", "list",
        "--location", RESOURCE_LOCATION,
        "--subscription", SUBSCRIPTION_ID,
    )
    quota = next(
        (item for item in usage if item.get("name", {}).get("value") == GRADER_QUOTA_NAME),
        None,
    )
    if not quota:
        raise RuntimeError(f"Quota {GRADER_QUOTA_NAME} is unavailable in {RESOURCE_LOCATION}.")

    existing = get_grader_deployment(account_name)
    existing_capacity = int((existing or {}).get("sku", {}).get("capacity") or 0)
    target_capacity = choose_grader_capacity(
        quota["limit"],
        quota["currentValue"],
        existing_capacity,
        GRADER_CAPACITY_SETTING,
    )
    return {
        "account_name": account_name,
        "deployment_name": GRADER_DEPLOYMENT,
        "model": GRADER_MODEL,
        "model_version": GRADER_MODEL_VERSION,
        "sku": GRADER_DEPLOYMENT_SKU,
        "quota_limit": int(quota["limit"]),
        "quota_used": int(quota["currentValue"]),
        "existing_capacity": existing_capacity,
        "target_capacity": target_capacity,
    }


def ensure_grader_deployment():
    plan = plan_grader_deployment()
    account_name = plan["account_name"]
    existing = get_grader_deployment(account_name)

    if grader_job_is_recorded():
        if not existing:
            raise RuntimeError(
                "The recorded model-grader job requires the configured grader deployment, "
                "but that deployment no longer exists. Reconcile it before resuming."
            )
        token_limit = deployment_token_limit(existing)
        if token_limit < GRADER_MIN_TPM:
            raise RuntimeError(
                f"The recorded job's grader deployment has {token_limit:,} TPM; "
                f"at least {GRADER_MIN_TPM:,} TPM is required. "
                "Capacity was not changed during resume."
            )
        print(
            f"Recorded model-grader job found; verified {GRADER_DEPLOYMENT} "
            f"at {token_limit:,} TPM without changing capacity."
        )
        return existing

    print(json.dumps({"grader_deployment_plan": plan}, indent=2))
    if not ALLOW_GRADER_DEPLOYMENT_UPDATE:
        raise RuntimeError(
            "Review the grader deployment plan, then set "
            "FOUNDRY_ALLOW_GRADER_DEPLOYMENT_UPDATE=true to authorize the ARM update."
        )

    deployment = run_az_json(
        "cognitiveservices", "account", "deployment", "create",
        "--name", account_name,
        "--resource-group", RESOURCE_GROUP,
        "--subscription", SUBSCRIPTION_ID,
        "--deployment-name", GRADER_DEPLOYMENT,
        "--model-name", GRADER_MODEL,
        "--model-version", GRADER_MODEL_VERSION,
        "--model-format", "OpenAI",
        "--sku-name", GRADER_DEPLOYMENT_SKU,
        "--sku-capacity", str(plan["target_capacity"]),
    )
    validate_existing_grader(deployment)
    token_limit = deployment_token_limit(deployment)
    if token_limit < GRADER_MIN_TPM:
        raise RuntimeError(
            f"Grader deployment reports {token_limit:,} TPM after the update; "
            f"at least {GRADER_MIN_TPM:,} TPM is required."
        )

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    evidence = {
        "recorded_at": datetime.now(timezone.utc).isoformat(),
        "account_name": account_name,
        "deployment_name": GRADER_DEPLOYMENT,
        "model": GRADER_MODEL,
        "model_version": GRADER_MODEL_VERSION,
        "sku": GRADER_DEPLOYMENT_SKU,
        "previous_capacity": plan["existing_capacity"],
        "requested_capacity": plan["target_capacity"],
        "reported_tpm": token_limit,
    }
    GRADER_DEPLOYMENT_STATE_PATH.write_text(
        json.dumps(evidence, indent=2) + "\n",
        encoding="utf-8",
    )
    print(
        f"Grader deployment ready: capacity {plan['target_capacity']} "
        f"({token_limit:,} TPM)."
    )
    return deployment


credential = project_client = openai_client = None
if not PROJECT_ENDPOINT:
    raise RuntimeError("Set FOUNDRY_PROJECT_ENDPOINT in .env or the process environment.")
grader_deployment = ensure_grader_deployment()
credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    allow_preview=True,
)
openai_client = project_client.get_openai_client()

In [ ]:
def wait_for_file(file_id, timeout_seconds=600):
    deadline = time.monotonic() + timeout_seconds
    delay = 1
    while True:
        try:
            uploaded = openai_client.files.retrieve(file_id)
        except Exception as exc:
            if getattr(exc, "status_code", None) != 404 or time.monotonic() >= deadline:
                raise
            print(f"File {file_id} is propagating; retrying in {delay}s")
            time.sleep(delay)
            delay = min(delay * 2, 10)
            continue
        status = str(uploaded.status).casefold()
        if status in {"processed", "completed"}:
            return uploaded
        if status in {"error", "failed", "expired", "cancelled"}:
            raise RuntimeError(f"File {file_id} ended as {status}: {uploaded}")
        if time.monotonic() >= deadline:
            raise TimeoutError(f"File {file_id} exceeded {timeout_seconds} seconds")
        time.sleep(5)


def upload_or_reuse_exact(label):
    contract = DATASETS[label]
    content = contract["path"].read_bytes().replace(b"\r\n", b"\n")
    remote_name = (
        f"countdown-{label}-{contract['rows']}-"
        f"{contract['sha256'][:12]}.jsonl"
    )
    for remote_file in openai_client.files.list(limit=100):
        if remote_file.filename != remote_name:
            continue
        try:
            wait_for_file(remote_file.id, timeout_seconds=15)
            remote_content = openai_client.files.content(remote_file.id).read()
        except Exception as exc:
            if getattr(exc, "status_code", None) == 404 or isinstance(exc, TimeoutError):
                print("Ignoring orphaned file listing:", remote_file.filename, remote_file.id)
                continue
            raise
        remote_sha256 = hashlib.sha256(remote_content).hexdigest()
        if remote_sha256 != contract["sha256"]:
            raise RuntimeError(f"Remote file {remote_name} has unexpected bytes")
        print("Reusing exact file:", remote_file.filename, remote_file.id)
        return remote_file.id
    uploaded = openai_client.files.create(
        file=(remote_name, io.BytesIO(content), "application/jsonl"),
        purpose="fine-tune",
    )
    wait_for_file(uploaded.id)
    print("Uploaded:", remote_name, uploaded.id)
    return uploaded.id


training_file_id = validation_file_id = None
training_file_id = upload_or_reuse_exact("training")
validation_file_id = upload_or_reuse_exact("validation")


## 5. Define the training recipe and resumable submission

Both variants use evaluation every 5 steps with 2 samples, medium reasoning effort, 1 epoch, batch size 4, and learning-rate multiplier 2.

In [ ]:
HYPERPARAMETERS = {
    "eval_interval": 5,
    "eval_samples": 2,
    "reasoning_effort": "medium",
    "n_epochs": 1,
    "batch_size": 4,
    "learning_rate_multiplier": 2,
}
TERMINAL_STATUSES = {"succeeded", "failed", "cancelled"}


def load_state():
    if not STATE_PATH.exists():
        return {"fingerprint": None, "jobs": {}}
    return json.loads(STATE_PATH.read_text(encoding="utf-8"))


def save_state(state):
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    STATE_PATH.write_text(
        json.dumps(state, indent=2, default=str) + "\n",
        encoding="utf-8",
    )


RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(
        {
            "endpoint": PROJECT_ENDPOINT,
            "model": MODEL,
            "model_version": MODEL_VERSION,
            "training_type": TRAINING_TYPE,
            "training_sha256": DATASETS["training"]["sha256"],
            "validation_sha256": DATASETS["validation"]["sha256"],
            "hyperparameters": HYPERPARAMETERS,
            "model_grader": MODEL_GRADER,
            "python_grader": PYTHON_GRADER,
            "response_format": RESPONSE_FORMAT,
        },
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()


def submit_once(label, grader, suffix):
    state = load_state()
    if state["fingerprint"] not in {None, RUN_FINGERPRINT}:
        raise RuntimeError(
            "The existing output state belongs to a different recipe. "
            "Move it aside before intentionally starting another run."
        )
    state["fingerprint"] = RUN_FINGERPRINT
    existing_job_id = state["jobs"].get(label)
    if existing_job_id:
        job = openai_client.fine_tuning.jobs.retrieve(existing_job_id)
        print(f"Resuming {label}: {job.id} ({job.status})")
        return job

    method = {
        "type": "reinforcement",
        "reinforcement": {
            "hyperparameters": HYPERPARAMETERS,
            "grader": grader,
            "response_format": RESPONSE_FORMAT,
        },
    }
    job = openai_client.with_options(max_retries=0).fine_tuning.jobs.create(
        model=MODEL,
        training_file=training_file_id,
        validation_file=validation_file_id,
        method=method,
        suffix=suffix,
        extra_body={"trainingType": TRAINING_TYPE},
    )
    state["jobs"][label] = job.id
    save_state(state)
    print(f"Submitted {label}: {job.id}")
    return job


def wait_for_terminal(label, job_id, poll_seconds=60, timeout_seconds=21_600):
    previous_status = None
    deadline = time.monotonic() + timeout_seconds
    while True:
        if time.monotonic() >= deadline:
            raise TimeoutError(f"{label} exceeded {timeout_seconds} seconds; rerun to resume.")
        try:
            job = openai_client.fine_tuning.jobs.retrieve(job_id)
        except Exception as exc:
            status_code = getattr(exc, "status_code", None)
            retryable = (
                status_code == 429
                or (status_code is not None and status_code >= 500)
                or type(exc).__name__
                in {
                    "APIConnectionError",
                    "APITimeoutError",
                    "ConnectError",
                    "ConnectTimeout",
                    "CredentialUnavailableError",
                    "ReadTimeout",
                    "TimeoutExpired",
                }
            )
            if not retryable:
                raise
            print(f"{label}: transient polling error; retrying: {exc}")
            time.sleep(poll_seconds)
            continue
        if job.status != previous_status:
            print(
                f"[{datetime.now(timezone.utc).isoformat()}] "
                f"{label}: {job.id} -> {job.status}"
            )
            previous_status = job.status
        if job.status in TERMINAL_STATUSES:
            if job.status != "succeeded":
                raise RuntimeError(f"{label} ended as {job.status}: {job.error}")
            return job
        time.sleep(poll_seconds)


def download_results_and_summarize(label, job):
    result_root = OUTPUT_ROOT / label
    result_root.mkdir(parents=True, exist_ok=True)
    rows = []
    files = []
    for index, file_id in enumerate(job.result_files or [], start=1):
        content = openai_client.files.content(file_id).read()
        path = result_root / f"result-{index}.csv"
        path.write_bytes(content)
        parsed = list(csv.DictReader(io.StringIO(content.decode("utf-8-sig"))))
        rows.extend(parsed)
        files.append(
            {
                "file_id": file_id,
                "path": str(path.relative_to(DEMO_ROOT)).replace("\\", "/"),
                "bytes": len(content),
                "sha256": hashlib.sha256(content).hexdigest(),
                "rows": len(parsed),
            }
        )

    validation_columns = [
        column
        for column in (rows[0].keys() if rows else [])
        if "reward" in column.lower()
        and ("valid" in column.lower() or "validation" in column.lower())
        and "error" not in column.lower()
    ]
    curve = []
    for row in rows:
        for column in validation_columns:
            value = row.get(column)
            if value not in {None, ""}:
                curve.append(
                    {
                        "step": int(float(row["step"])) if row.get("step") else None,
                        "column": column,
                        "value": float(value),
                    }
                )
                break
    values = [point["value"] for point in curve]
    if not values:
        raise RuntimeError(f"{label}: no service validation reward metrics returned")
    summary = {
        "label": label,
        "job_id": job.id,
        "status": job.status,
        "model": job.model,
        "training_type": TRAINING_TYPE,
        "trained_tokens": job.trained_tokens,
        "error": job.error.model_dump(mode="json") if job.error else None,
        "input_hashes": {
            key: DATASETS[key]["sha256"]
            for key in ("training", "validation", "evaluation_provenance")
        },
        "input_evidence": {
            key: {
                "raw_bytes": len(DATASETS[key]["path"].read_bytes()),
                "raw_sha256": hashlib.sha256(
                    DATASETS[key]["path"].read_bytes()
                ).hexdigest(),
                "normalized_bytes": len(
                    DATASETS[key]["path"].read_bytes().replace(b"\r\n", b"\n")
                ),
                "normalized_sha256": DATASETS[key]["sha256"],
            }
            for key in ("training", "validation", "evaluation_provenance")
        },
        "result_files": files,
        "validation_curve": curve,
        "validation_reward_initial": values[0] if values else None,
        "validation_reward_final": values[-1] if values else None,
        "validation_reward_max": max(values) if values else None,
    }
    (result_root / "summary.json").write_text(
        json.dumps(summary, indent=2, default=str) + "\n",
        encoding="utf-8",
    )
    print(json.dumps(summary, indent=2, default=str))
    return summary

## 6. Submit or resume both jobs

Job-creation POSTs have SDK retries disabled. If a submission response is ambiguous, inspect the project before rerunning so an existing job is not duplicated.

In [ ]:
model_grader_job = submit_once(
    label="model-grader",
    grader=MODEL_GRADER,
    suffix="qwen36-countdown-model-grader",
)
python_grader_job = submit_once(
    label="python-grader",
    grader=PYTHON_GRADER,
    suffix="qwen36-countdown-python-grader",
)


## 7. Monitor the model/score-grader job and download its result CSV

In [ ]:
model_grader_job = wait_for_terminal("model-grader", model_grader_job.id)
model_grader_summary = download_results_and_summarize(
    "model-grader",
    model_grader_job,
)


## 8. Monitor the deterministic Python-grader job and download its result CSV

In [ ]:
python_grader_job = wait_for_terminal("python-grader", python_grader_job.id)
python_grader_summary = download_results_and_summarize(
    "python-grader",
    python_grader_job,
)


## 9. Reward-curve summary

In [ ]:
reward_summary = {
    "model-grader": {
        "status": model_grader_summary["status"],
        "initial": model_grader_summary["validation_reward_initial"],
        "final": model_grader_summary["validation_reward_final"],
        "max": model_grader_summary["validation_reward_max"],
    },
    "python-grader": {
        "status": python_grader_summary["status"],
        "initial": python_grader_summary["validation_reward_initial"],
        "final": python_grader_summary["validation_reward_final"],
        "max": python_grader_summary["validation_reward_max"],
    },
}
print(json.dumps(reward_summary, indent=2))
openai_client.close()
project_client.close()
credential.close()


### Interpret the training metrics

Use the generated `reward_summary` and the saved result CSVs to inspect each grader's validation-reward trajectory. These values are service training signals for this run; they do not establish held-out task quality or deployment behavior.

The grader deployment is not automatically resized or deleted. After both jobs are terminal, review `outputs/grader-deployment-state.json` and deliberately restore the prior capacity or remove the deployment through your approved resource-management process if it is no longer needed.